# `Create Database for MIMIC Dataset`

Creates SQLite dataset for MIMIC-IV v3.1 

| **Done** | **Task**  |
|----------|-----------|
| [X] | Access Data Files |
| [X] | Design Databases |
| [X] | Read Aquisition Parameters |




## 1. `Define Data Files`

### Workflow Annotation:

- Read the configuration file containing the path to the folder where the MIMIC dataset CSV files are stored.
- Load the settings from the configuration file.
- Verify and adjust the dataset path if necessary.
- Check the existence of all required CSV files in the specified folder.


In [1]:
from config.settings_data import DataSetsSettings

profile = "default"

settings = DataSetsSettings.build(profile)
#print(settings.MIMIC_IV_root)


In [2]:
'''
Check existance of all required CSV files.
'''
from utils import templates as T
from config.settings_data import load_csv_settings
from utils.dataset_utils import check_MIMIC_IV_csv_file_existance

csv_settings = load_csv_settings( profile )
report = check_MIMIC_IV_csv_file_existance( csv_settings, T )



Number of CSV files checked: 37


## 2.1 `Collect DICOM Meta-Data`


In [ ]:
from utils.dataset_utils import generate_image_acquisition_parameter_csv


generate_image_acquisition_parameter_csv(
    settings.MIMIC_CXR_root,
    csv_settings.images_csv,
    csv_settings.image_acquisition_csv
)


## 2.2 `Format MIMIC-CXR-JPG-CheXpert Diagnostic Data`


In [ ]:
from utils.dataset_utils import format_mimic_chexpert_diagnostic_csv

format_mimic_chexpert_diagnostic_csv(
    source_csv_path = csv_settings.source_chexpert_diagnosis_csv,
    target_csv_path = csv_settings.formated_chexpert_diagnosis_csv
    )







In [ ]:
from utils.dataset_utils import create_studies_csv

create_studies_csv(
    input_csv_path = csv_settings.source_chexpert_diagnosis_csv,
    output_csv_path = csv_settings.studies_csv
)

# 2.3 `Create Databese: HOSP module`



| **Done** | **Table Name**        | **Description** |
|-----------|-----------------------|-----------------|
| [X] | **omr**                | The Online Medical Record (OMR) table contains miscellaneous information from the EHR. |
| [X] | **provider**           | The provider table lists deidentified provider identifiers used in the database. |
| [X] | **admissions**         | Detailed information about hospital stays. |
| [X] | **d_hcpcs**            | Dimension table for hcpcsevents; provides a description of CPT codes. |
| [X] | **d_icd_diagnoses**    | Dimension table for diagnoses_icd; provides a description of ICD-9/ICD-10 billed diagnoses. |
| [X] | **d_icd_procedures**   | Dimension table for procedures_icd; provides a description of ICD-9/ICD-10 billed procedures. |
| [X] | **d_labitems**         | Dimension table for labevents; provides a description of all lab items. |
| [X] | **diagnoses_icd**      | Billed ICD-9/ICD-10 diagnoses for hospitalizations. |
| [X] | **drgcodes**           | Billed diagnosis related group (DRG) codes for hospitalizations. |
| [X] | **emar**               | The Electronic Medicine Administration Record (eMAR); barcode scanning of medications at the time of administration. |
| [X] | **emar_detail**        | Supplementary information for electronic administrations recorded in emar. |
| [X] | **hcpcsevents**        | Billed events occurring during the hospitalization. Includes CPT codes. |
| [X] | **labevents**          | Laboratory measurements sourced from patient derived specimens. |
| [X] | **microbiologyevents** | Microbiology cultures. |
| [X] | **patients**           | Patients' gender, age, and date of death if information exists. |
| [X] | **pharmacy**           | Formulary, dosing, and other information for prescribed medications. |
| [X] | **poe**                | Orders made by providers relating to patient care. |
| [X] | **poe_detail**         | Supplementary information for orders made by providers in the hospital. |
| [X] | **prescriptions**      | Prescribed medications. |
| [X] | **procedures_icd**     | Billed procedures for patients during their hospital stay. |
| [X] | **services**           | The hospital service(s) which cared for the patient during their hospitalization. |
| [X] | **transfers**          | Detailed information about patients' unit transfers. |

#### ICU Tables


| **Done** | **Table Name**        | **Description** |
|-----------|-----------------------|-----------------|
| [X] | **caregiver**                | The caregiver table lists deidentified provider identifiers used in the ICU module. |
| [X] | **d_items**           | Dimension table describing itemid. Defines concepts recorded in the events table in the ICU module. |
| [X] | **chartevents**         | Charted items occurring during the ICU stay. Contains the majority of information documented in the ICU. |
| [X] | **datetimeevents**            | Documented information which is in a date format (e.g. date of last dialysis). |
| [X] | **ICU stays**    | Tracking information for ICU stays including admission and discharge times. |
| [X] | **Ingredientevents**   | Ingredients of continuous or intermittent administrations including nutritional and water content. |
| [X] | **Inputevents**         | Information documented regarding continuous infusions or intermittent administrations. |
| [X] | **outputevents**      | Information regarding patient outputs including urine, drainage, and so on. |
| [X] | **procedureevents**           | Procedures documented during the ICU stay (e.g. ventilation), though not necessarily conducted within the ICU (e.g. x-ray imaging). |


In [3]:
from utils.dataset_utils import create_database_from_sql

create_database_from_sql(
    db_path = settings.DB_name,
    schema_path = settings.SQL_script
)

# 2.4 `Data Correction`


In [4]:
from utils.mimic_dataset_corrections import *

#clean_microbiologyevents_csv_inplace(
#    csv_settings.microbiologyevents_csv
#)

#clean_prescriptions_required_columns_inplace(
#    csv_settings.prescriptions_csv
#)

missing_cxr_pid = check_csv_study_id_match(
    csv_settings.patients_csv,
    csv_settings.studies_csv,
    csv_settings.missing_cxr_pid_csv
)

insert_missing_subject_ids(
    settings.DB_name,
    csv_settings.missing_cxr_pid_csv
)

There are 3511 missing PIDs


# 2.5 `Inject Data`


In [5]:
from utils.dataset_utils import get_csv_table_mapping, inject_csv_to_table, drop_table_data

TABLE_INSERT_ORDER, TABLE_TO_CSV = get_csv_table_mapping( csv_settings )
#start_with = TABLE_INSERT_ORDER[0]
#start_with = 'studies'
#started_status = False
#start_idx = TABLE_INSERT_ORDER.index(start_with)

#for table_name in TABLE_INSERT_ORDER[start_idx:]:
for table_name in TABLE_INSERT_ORDER:
    print('Processing: '+table_name)
#    if not started_status:
#        drop_table_data(
#            settings.DB_name,
#            table_name
#        )
#        started_status = True

    csv_path = TABLE_TO_CSV[table_name]
    inject_csv_to_table(
        settings.DB_name,
        csv_path,
        table_name
    )


Processing: patients
Processing: provider
Processing: d_hcpcs
Processing: d_icd_diagnoses
Processing: d_icd_procedures
Processing: d_labitems
Processing: admissions
Processing: omr
Processing: hcpcsevents
Processing: diagnoses_icd
Processing: procedures_icd
Processing: labevents
Processing: microbiologyevents
Processing: drgcodes
Processing: services
Processing: transfers
Processing: poe
Processing: poe_detail
Processing: pharmacy
Processing: prescriptions
Processing: emar
Processing: emar_detail
Processing: studies
Processing: images
Processing: image_acquisition
Processing: chexpert_diagnosis


Injection issues:

1. NOT NULL constraint failed: microbiologyevents.spec_type_desc ->
Backup created: D:\003.Data\MIMIC-IV.v3.1\physionet.org\files\mimiciv\3.1\hosp\microbiologyevents.csv.bak
Cleaned file overwritten: D:\003.Data\MIMIC-IV.v3.1\physionet.org\files\mimiciv\3.1\hosp\microbiologyevents.csv
Total rows: 3988224
Removed rows: 1
Kept rows: 3988223

2. NOT NULL constraint failed: prescriptions.drug ->

3. MIMIC-IV does not provide information for 3511 patients in MIMIC-CXR 

